# 00 · Verificación del entorno reproducible

Proyecto: precipitación del IDEAM (`s54a-sgyg`) · Juan Pablo Castro

Este cuaderno responde tres preguntas, en orden. Si alguna falla, el entorno no está listo y no vale la pena seguir:

1. ¿El intérprete y las librerías son los que el proyecto ancló?
2. ¿Los volúmenes del anfitrión están montados dentro del contenedor?
3. ¿El servicio de Jupyter alcanza la base de datos **por su nombre de servicio**?


## 1. Intérprete y librerías ancladas

In [1]:
import sys
import platform
from importlib import metadata

print("Python  :", sys.version.split()[0])
print("Sistema :", platform.platform())
print("Máquina :", platform.machine())


Python  : 3.12.11
Sistema : Linux-6.12.76-linuxkit-aarch64-with-glibc2.39
Máquina : aarch64


In [2]:
# Las versiones esperadas se leen del propio requirements.txt:
# así una sola fuente de verdad gobierna el anclaje y esta celda
# no puede quedar desincronizada del archivo.
import re
from pathlib import Path

REQUIREMENTS = Path("/tmp/requirements.txt")

expected = {}
for line in REQUIREMENTS.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    match = re.match(r"^([A-Za-z0-9_.\-]+)(?:\[[^\]]+\])?==(.+)$", line)
    if match:
        expected[match.group(1).lower()] = match.group(2).strip()

print(f"Paquetes anclados en requirements.txt: {len(expected)}\n")

discrepancies = []
for package, wanted in sorted(expected.items()):
    try:
        found = metadata.version(package)
    except metadata.PackageNotFoundError:
        found = None
    if found == wanted:
        status = "OK"
    elif found is None:
        status = "AUSENTE"
        discrepancies.append(package)
    else:
        status = "DISCREPANCIA"
        discrepancies.append(package)
    print(f"{status:13s} · {package}: esperado {wanted}, obtenido {found}")

print()
if discrepancies:
    print("Revise el anclaje de versiones:", ", ".join(discrepancies))
else:
    print("Entorno reproducible verificado: todas las versiones coinciden.")


Paquetes anclados en requirements.txt: 8

OK            · ipykernel: esperado 6.29.5, obtenido 6.29.5
OK            · jupyterlab: esperado 4.2.5, obtenido 4.2.5
OK            · numpy: esperado 1.26.4, obtenido 1.26.4
OK            · pandas: esperado 2.2.3, obtenido 2.2.3
OK            · psutil: esperado 7.2.2, obtenido 7.2.2
OK            · psycopg: esperado 3.2.3, obtenido 3.2.3
OK            · requests: esperado 2.32.5, obtenido 2.32.5
OK            · sqlalchemy: esperado 2.0.35, obtenido 2.0.35

Entorno reproducible verificado: todas las versiones coinciden.


## 2. Volúmenes montados

Los datos y el código viven en el anfitrión y se **montan** en el contenedor. Si esta celda no ve las carpetas, el `docker-compose.yml` no está montando lo que cree.


In [3]:
from pathlib import Path

WORK = Path("/home/jovyan/work")
for name in ["notebooks", "src", "data", "data/raw", "sql", "docs"]:
    path = WORK / name
    mark = "OK   " if path.is_dir() else "FALTA"
    print(f"{mark} · {path}")

raw = WORK / "data" / "raw"
if raw.is_dir():
    files = sorted(p for p in raw.iterdir() if p.name != ".gitkeep")
    print(f"\nArchivos en data/raw: {len(files)}")
    for path in files:
        print(f"  {path.name}  ({path.stat().st_size:,} bytes)")
    if not files:
        print("  (vacío: es lo esperado en un clon limpio;")
        print("   el README explica cómo obtener las particiones)")


OK    · /home/jovyan/work/notebooks
OK    · /home/jovyan/work/src
OK    · /home/jovyan/work/data
OK    · /home/jovyan/work/data/raw
OK    · /home/jovyan/work/sql
OK    · /home/jovyan/work/docs

Archivos en data/raw: 2
  precipitacion_2026-06-21.csv  (21,762,351 bytes)
  precipitacion_2026-06-22.csv  (21,953,076 bytes)


## 3. Conexión a PostgreSQL por nombre de servicio

Esta es la celda que prueba que se entendió la red interna de Compose. La conexión apunta a **`db`**, el nombre del servicio, no a `localhost` ni a una dirección IP fija.

Las credenciales llegan por variables de entorno inyectadas desde `.env`. No hay contraseñas escritas en este cuaderno.


In [4]:
import os

import psycopg

# El host es el NOMBRE DEL SERVICIO del docker-compose.yml.
# Docker resuelve "db" dentro de la red que crea para el proyecto.
connection_parameters = {
    "host": os.environ["POSTGRES_HOST"],
    "port": os.environ["POSTGRES_PORT"],
    "dbname": os.environ["POSTGRES_DB"],
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"],
}

print("Host de conexión:", connection_parameters["host"], "(nombre de servicio)")
print("Base de datos   :", connection_parameters["dbname"])
print("Usuario         :", connection_parameters["user"])
print("Contraseña      : (leída de la variable de entorno, no se imprime)")


Host de conexión: db (nombre de servicio)
Base de datos   : ideam
Usuario         : ideam_app
Contraseña      : (leída de la variable de entorno, no se imprime)


In [5]:
with psycopg.connect(**connection_parameters) as connection:
    with connection.cursor() as cursor:
        cursor.execute("SELECT version();")
        engine_version = cursor.fetchone()[0]

        cursor.execute("SELECT current_database(), current_user;")
        database, user = cursor.fetchone()

        cursor.execute(
            """
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
            ORDER BY table_name;
            """
        )
        tables = [row[0] for row in cursor.fetchall()]

print("Motor:", engine_version)
print()
print("Base conectada:", database, "· usuario:", user)
print("Tablas creadas por sql/01_esquema.sql:", tables or "(ninguna)")


Motor: PostgreSQL 16.4 on aarch64-unknown-linux-musl, compiled by gcc (Alpine 13.2.1_git20240309) 13.2.1 20240309, 64-bit

Base conectada: ideam · usuario: ideam_app
Tablas creadas por sql/01_esquema.sql: ['control_ingesta', 'precipitacion']


### Comprobación del esquema

Se verifica que la clave primaria de `precipitacion` sea exactamente la clave candidata que T1 validó sobre los datos: `codigoestacion + codigosensor + fechaobservacion`.


In [6]:
EXPECTED_KEY = ["codigoestacion", "codigosensor", "fechaobservacion"]

with psycopg.connect(**connection_parameters) as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT kcu.column_name
            FROM information_schema.table_constraints AS tc
            JOIN information_schema.key_column_usage AS kcu
              ON tc.constraint_name = kcu.constraint_name
            WHERE tc.table_name = 'precipitacion'
              AND tc.constraint_type = 'PRIMARY KEY'
            ORDER BY kcu.ordinal_position;
            """
        )
        primary_key = [row[0] for row in cursor.fetchall()]

print("Clave primaria en la base:", primary_key)
print("Clave candidata de T1    :", EXPECTED_KEY)
print()
print(
    "OK · la restricción de la base reproduce el hallazgo de T1."
    if primary_key == EXPECTED_KEY
    else "DISCREPANCIA · revise sql/01_esquema.sql"
)


Clave primaria en la base: ['codigoestacion', 'codigosensor', 'fechaobservacion']
Clave candidata de T1    : ['codigoestacion', 'codigosensor', 'fechaobservacion']

OK · la restricción de la base reproduce el hallazgo de T1.


## Resultado

Si las tres secciones anteriores no reportan discrepancias, el entorno está listo y se levantó **sin intervención manual** más allá de copiar `.env.example` a `.env`.

Ese es el criterio de aceptación de T2: un clon limpio corre con un solo comando.
